### Image Conversion and Dataset Preparation for KAIR

In [ ]:
import os
import random
import rasterio
import numpy as np
from pathlib import Path
from PIL import Image

# ==========================================
# Configuration
# ==========================================
SOURCE_DIR = r"D:\SUPARCO\SEN2VENµS"

# Configurable destinations for Train and Test sets
DEST_TRAIN_DIR = r"./"
DEST_TEST_DIR = r"../../testsets/Sen2Venus"

# Split ratio (e.g., 0.8 = 80% train, 20% test per location)
TRAIN_RATIO = 0.9
RANDOM_SEED = 42  # For reproducible splits

# Configurable clipping maximum value
CLIP_MAX = 2500

# ==========================================
# Core Processing Functions
# ==========================================

def norm_clip(img_array, clip_max):
    """Method 1: Clip negatives to 0, stretch max to clip_max."""
    # Stack channels to RGB (Bands 2, 1, 0 are Red, Green, Blue in Sen2Venus)
    rgb = np.stack((img_array[2], img_array[1], img_array[0]), axis=-1).astype(np.float32)
    rgb = np.clip(rgb, 0, clip_max)
    return rgb / float(clip_max)

def process_and_save_image(tiff_path, save_path, clip_max):
    """Reads a TIFF, normalizes it, and saves it as an 8-bit PNG."""
    try:
        with rasterio.open(tiff_path) as src:
            img_array = src.read()

        # 1. Apply your exact normalization (returns float32 between 0.0 and 1.0)
        rgb_norm = norm_clip(img_array, clip_max)

        # 2. Convert to 8-bit unsigned integer (0-255) required for PNGs
        rgb_8bit = (rgb_norm * 255.0).astype(np.uint8)

        # 3. Save as PNG
        Image.fromarray(rgb_8bit).save(save_path)
        return True
    except Exception as e:
        print(f"  [Error] Failed to process {tiff_path.name}: {e}")
        return False

# ==========================================
# Main Pipeline
# ==========================================

def prepare_kair_dataset(source_dir, dest_train, dest_test, train_ratio=0.8, seed=42, clip_max=2500):
    random.seed(seed)
    source_path = Path(source_dir)

    # Setup destination paths
    paths = {
        'train_hr': Path(dest_train) / "HR",
        'train_lr': Path(dest_train) / "LR",
        'test_hr': Path(dest_test) / "HR",
        'test_lr': Path(dest_test) / "LR"
    }

    # Create all necessary directories
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)

    locations = [d for d in source_path.iterdir() if d.is_dir()]

    total_train, total_test = 0, 0

    print(f"{'='*50}\nStarting Dataset Preparation\n{'='*50}")

    for loc_dir in locations:
        loc_name = loc_dir.name
        dir_5m = loc_dir / 'b2b3b4b8' / '05m'
        dir_10m = loc_dir / 'b2b3b4b8' / '10m'

        # Skip if folder structure doesn't exist for this location
        if not dir_5m.exists() or not dir_10m.exists():
            continue

        # Gather all valid matching pairs for this location
        valid_pairs = []
        for hr_img_path in dir_5m.glob('*.tif*'):
            hr_filename = hr_img_path.name
            lr_filename = hr_filename.replace('_05m', '_10m')
            lr_img_path = dir_10m / lr_filename

            if lr_img_path.exists():
                valid_pairs.append((hr_img_path, lr_img_path))

        if not valid_pairs:
            continue

        # Stratified Splitting: Shuffle and split *within* the current location
        random.shuffle(valid_pairs)
        split_idx = int(len(valid_pairs) * train_ratio)

        # Ensure at least 1 image goes to train and 1 to test if there are >= 2 images
        if 0 < split_idx < len(valid_pairs) - 1:
            pass # normal split
        elif len(valid_pairs) >= 2:
            split_idx = max(1, min(split_idx, len(valid_pairs) - 1))

        train_pairs = valid_pairs[:split_idx]
        test_pairs = valid_pairs[split_idx:]

        print(f"Location: {loc_name:15} | Total: {len(valid_pairs):3} | Train: {len(train_pairs):3} | Test: {len(test_pairs):3}")

        # Process the split
        for subset_name, pairs, dest_hr, dest_lr in [
            ("Train", train_pairs, paths['train_hr'], paths['train_lr']),
            ("Test", test_pairs, paths['test_hr'], paths['test_lr'])
        ]:
            for hr_path, lr_path in pairs:
                # Create exact matching filename for KAIR
                # Strip '_05m.tif' / '_05m.tiff' and append '.png'
                base_name = hr_path.name.replace('_05m.tif', '').replace('_05m.tiff', '')
                base_name = base_name.replace('.tif', '').replace('.tiff', '') # Safety catch
                save_name = f"{base_name}.png"

                out_hr_path = dest_hr / save_name
                out_lr_path = dest_lr / save_name

                # Process and save
                success_hr = process_and_save_image(hr_path, out_hr_path, clip_max)
                success_lr = process_and_save_image(lr_path, out_lr_path, clip_max)

                if success_hr and success_lr:
                    if subset_name == "Train":
                        total_train += 1
                    else:
                        total_test += 1

    print(f"\n{'='*50}")
    print("Processing Complete!")
    print(f"Total Train Image Pairs: {total_train}")
    print(f"Total Test Image Pairs:  {total_test}")
    print(f"{'='*50}")

# ==========================================
# Execution
# ==========================================
if __name__ == "__main__":
    prepare_kair_dataset(
        source_dir=SOURCE_DIR,
        dest_train=DEST_TRAIN_DIR,
        dest_test=DEST_TEST_DIR,
        train_ratio=TRAIN_RATIO,
        seed=RANDOM_SEED,
        clip_max=CLIP_MAX
    )

Starting Dataset Preparation
Location: ALSACE          | Total: 2653 | Train: 2387 | Test: 266
Location: ANJI            | Total: 2312 | Train: 2080 | Test: 232
Location: ARM             | Total: 15859 | Train: 14273 | Test: 1586
Location: ATTO            | Total: 2258 | Train: 2032 | Test: 226
Location: BAMBENW2        | Total: 9018 | Train: 8116 | Test: 902
Location: BENGA           | Total: 5857 | Train: 5271 | Test: 586
Location: ES-IC3XG        | Total: 8822 | Train: 7939 | Test: 883
Location: ES-LTERA        | Total: 1701 | Train: 1530 | Test: 171
Location: ESGISB-1        | Total: 2891 | Train: 2601 | Test: 290
Location: ESGISB-2        | Total: 3067 | Train: 2760 | Test: 307
Location: ESGISB-3        | Total: 6057 | Train: 5451 | Test: 606


### Resuming incase of failure

In [1]:
import os
import random
import rasterio
import numpy as np
from pathlib import Path
from PIL import Image
import gc  # Added for memory management

# ==========================================
# Configuration
# ==========================================
SOURCE_DIR = r"D:\SUPARCO\SEN2VENµS"

# Configurable destinations for Train and Test sets
DEST_TRAIN_DIR = r"./"
DEST_TEST_DIR = r"../../testsets/Sen2Venus"

# Split ratio (e.g., 0.8 = 80% train, 20% test per location)
TRAIN_RATIO = 0.9
RANDOM_SEED = 42  # For reproducible splits

# Configurable clipping maximum value
CLIP_MAX = 2500

# Set this to the name of the folder where it crashed, e.g., "NARYN"
# Set to None if you want to start from the very beginning.
RESUME_FROM_LOC = 'ESGISB-3'

# ==========================================
# Core Processing Functions
# ==========================================

def norm_clip(img_array, clip_max):
    """Method 1: Clip negatives to 0, stretch max to clip_max."""
    # Stack channels to RGB (Bands 2, 1, 0 are Red, Green, Blue in Sen2Venus)
    rgb = np.stack((img_array[2], img_array[1], img_array[0]), axis=-1).astype(np.float32)
    rgb = np.clip(rgb, 0, clip_max)
    return rgb / float(clip_max)

def process_and_save_image(tiff_path, save_path, clip_max):
    """Reads a TIFF, normalizes it, and saves it as an 8-bit PNG."""
    try:
        with rasterio.open(tiff_path) as src:
            img_array = src.read()

        # 1. Apply your exact normalization (returns float32 between 0.0 and 1.0)
        rgb_norm = norm_clip(img_array, clip_max)

        # 2. Convert to 8-bit unsigned integer (0-255) required for PNGs
        rgb_8bit = (rgb_norm * 255.0).astype(np.uint8)

        # 3. Save as PNG
        Image.fromarray(rgb_8bit).save(save_path)

        # Free memory explicitly
        del img_array, rgb_norm, rgb_8bit
        return True
    except Exception as e:
        print(f"  [Error] Failed to process {tiff_path.name}: {e}")
        return False

# ==========================================
# Main Pipeline
# ==========================================

def prepare_kair_dataset(source_dir, dest_train, dest_test, train_ratio=0.8, seed=42, clip_max=2500, resume_from=None):
    random.seed(seed)
    source_path = Path(source_dir)

    # Setup destination paths
    paths = {
        'train_hr': Path(dest_train) / "HR",
        'train_lr': Path(dest_train) / "LR",
        'test_hr': Path(dest_test) / "HR",
        'test_lr': Path(dest_test) / "LR"
    }

    # Create all necessary directories
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)

    # SORT the locations alphabetically so the order is always exactly the same
    locations = sorted([d for d in source_path.iterdir() if d.is_dir()], key=lambda x: x.name)

    total_train, total_test = 0, 0

    print(f"{'='*50}\nStarting Dataset Preparation\n{'='*50}")

    skipping_mode = bool(resume_from)

    for loc_dir in locations:
        loc_name = loc_dir.name

        # Resume Logic
        if skipping_mode:
            if loc_name == resume_from:
                print(f">>> Resuming processing at location: {loc_name} <<<")
                skipping_mode = False
            else:
                print(f"Skipping {loc_name} (already processed)...")
                continue

        dir_5m = loc_dir / 'b2b3b4b8' / '05m'
        dir_10m = loc_dir / 'b2b3b4b8' / '10m'

        # Skip if folder structure doesn't exist for this location
        if not dir_5m.exists() or not dir_10m.exists():
            continue

        # Gather all valid matching pairs for this location
        valid_pairs = []
        for hr_img_path in dir_5m.glob('*.tif*'):
            hr_filename = hr_img_path.name
            lr_filename = hr_filename.replace('_05m', '_10m')
            lr_img_path = dir_10m / lr_filename

            if lr_img_path.exists():
                valid_pairs.append((hr_img_path, lr_img_path))

        if not valid_pairs:
            continue

        # Stratified Splitting: Shuffle and split *within* the current location
        random.shuffle(valid_pairs)
        split_idx = int(len(valid_pairs) * train_ratio)

        # Ensure at least 1 image goes to train and 1 to test if there are >= 2 images
        if 0 < split_idx < len(valid_pairs) - 1:
            pass # normal split
        elif len(valid_pairs) >= 2:
            split_idx = max(1, min(split_idx, len(valid_pairs) - 1))

        train_pairs = valid_pairs[:split_idx]
        test_pairs = valid_pairs[split_idx:]

        print(f"Location: {loc_name:15} | Total: {len(valid_pairs):3} | Train: {len(train_pairs):3} | Test: {len(test_pairs):3}")

        # Process the split
        for subset_name, pairs, dest_hr, dest_lr in [
            ("Train", train_pairs, paths['train_hr'], paths['train_lr']),
            ("Test", test_pairs, paths['test_hr'], paths['test_lr'])
        ]:
            for hr_path, lr_path in pairs:
                # Create exact matching filename for KAIR
                base_name = hr_path.name.replace('_05m.tif', '').replace('_05m.tiff', '')
                base_name = base_name.replace('.tif', '').replace('.tiff', '') # Safety catch
                save_name = f"{base_name}.png"

                out_hr_path = dest_hr / save_name
                out_lr_path = dest_lr / save_name

                # Process and save
                success_hr = process_and_save_image(hr_path, out_hr_path, clip_max)
                success_lr = process_and_save_image(lr_path, out_lr_path, clip_max)

                if success_hr and success_lr:
                    if subset_name == "Train":
                        total_train += 1
                    else:
                        total_test += 1

        # Force Python to clean up memory after every location folder
        gc.collect()

    print(f"\n{'='*50}")
    print("Processing Complete!")
    print(f"Total Train Image Pairs (this run): {total_train}")
    print(f"Total Test Image Pairs (this run):  {total_test}")
    print(f"{'='*50}")

# ==========================================
# Execution
# ==========================================
if __name__ == "__main__":
    prepare_kair_dataset(
        source_dir=SOURCE_DIR,
        dest_train=DEST_TRAIN_DIR,
        dest_test=DEST_TEST_DIR,
        train_ratio=TRAIN_RATIO,
        seed=RANDOM_SEED,
        clip_max=CLIP_MAX,
        resume_from=RESUME_FROM_LOC
    )

Starting Dataset Preparation
Skipping ALSACE (already processed)...
Skipping ANJI (already processed)...
Skipping ARM (already processed)...
Skipping ATTO (already processed)...
Skipping BAMBENW2 (already processed)...
Skipping BENGA (already processed)...
Skipping ES-IC3XG (already processed)...
Skipping ES-LTERA (already processed)...
Skipping ESGISB-1 (already processed)...
Skipping ESGISB-2 (already processed)...
>>> Resuming processing at location: ESGISB-3 <<<
Location: ESGISB-3        | Total: 6057 | Train: 5451 | Test: 606
Location: ESTUAMAR        | Total: 911 | Train: 819 | Test:  92
Location: FGMANAUS        | Total: 129 | Train: 116 | Test:  13
Location: FR-BIL          | Total: 7105 | Train: 6394 | Test: 711
Location: FR-LAM          | Total: 7299 | Train: 6569 | Test: 730
Location: FR-LQ1          | Total: 4888 | Train: 4399 | Test: 489
Location: JAM2018         | Total: 2564 | Train: 2307 | Test: 257
Location: K34-AMAZ        | Total: 1384 | Train: 1245 | Test: 139
Locat